# Boltz-2 Structure Prediction

![Boltz-2 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/boltz2/hero.png)

This notebook demonstrates multi-modal structure prediction using Boltz-2, an open-source model developed by MIT. Boltz-2 predicts the three-dimensional structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes using a diffusion-based generative approach with configurable recycling and sampling steps. We demonstrate `run_boltz2` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

## Available tools

In [ ]:
display_available_tools("boltz2")

### `run_boltz2`

Boltz-2 predicts 3D structures of proteins, DNA, RNA, ligands, and their complexes using a diffusion-based generative model. It supports both single-sequence mode and MSA-assisted prediction via ColabFold search. The `recycling_steps` parameter controls iterative structural refinement, while `sampling_steps` governs the granularity of the denoising process. When `diffusion_samples` is set above 1, Boltz-2 generates multiple independent structure samples and returns the best by confidence score, which is particularly useful for exploring conformational diversity in flexible protein-ligand systems. Ligands are provided as SMILES strings and are automatically converted to the appropriate internal representation.

In [ ]:
from pathlib import Path

from proto_tools import (
    Mmseqs2HomologySearchConfig,
    Boltz2Config,
    Boltz2Input,
    Chain,
    Complex,
    run_boltz2,
)

In [ ]:
# Display input docs
display_api_reference("boltz2", "input", "run_boltz2")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Boltz2Input(complexes=[complex])

In [ ]:
# Display config docs
display_api_reference("boltz2", "config", "run_boltz2")

# Configure Boltz-2 with reduced settings for a fast demonstration run
config = Boltz2Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    recycling_steps=3,
    diffusion_samples=1,
)

In [ ]:
# Run structure prediction
result = run_boltz2(inputs, config)

In [ ]:
# Display output docs
display_api_reference("boltz2", "output", "run_boltz2")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(mfng_sequence)} residues")
print(f"  Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"  Complex pLDDT:     {mfng_structure.metrics.complex_plddt:.3f}")
print(f"  pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"  Ligand ipTM:       {mfng_structure.metrics.ligand_iptm:.3f}")

#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain, allowing you to inspect the binding pose and overall fold geometry.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
mfng_structure.visualize(style="cartoon", color_by="bfactor")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print("Structure exported to", sorted((output_dir / "mfng_ligand_complex").glob("structure_*.pdb")))

## Predict Binding Affinity

Boltz-2's affinity head predicts the binding affinity of a single small-molecule ligand against a protein target. `run_boltz2_affinity` runs the same diffusion model with an added `properties.affinity.binder` block, so the predicted binding pose is returned alongside the affinity scores. The binder is auto-detected here because the complex has a single ligand; set `binder_chain` to name it when a complex has several. `binder_chain` is per-complex: pass one value (e.g. `["B"]`) to apply it to every complex, or one per complex (e.g. `["B", None]`, where `None` auto-detects).

In [ ]:
from proto_tools import Boltz2AffinityConfig, Boltz2AffinityInput, run_boltz2_affinity

display_api_reference("boltz2", "input", "run_boltz2_affinity")

# Reuse the MfnG protein + L-tyrosine complex from above (single ligand → binder auto-detected).
affinity_inputs = Boltz2AffinityInput(complexes=[complex])
affinity_config = Boltz2AffinityConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    diffusion_samples=1,
    sampling_steps_affinity=200,
    diffusion_samples_affinity=5,
)

affinity_result = run_boltz2_affinity(affinity_inputs, affinity_config)
affinity_structure = affinity_result.structures[0]
print(f"  affinity_pred_value:         {affinity_structure.metrics.affinity_pred_value:.3f}  (log10 IC50 μM; lower = stronger)")
print(f"  affinity_probability_binary: {affinity_structure.metrics.affinity_probability_binary:.3f}  (binder probability)")